## Setup


### Install the required packages.

In [9]:
!pip install -qU crewai[tools,agentops]==0.95.0

ERROR: Ignored the following yanked versions: 0.165.0, 1.10.0, 1.12.0, 1.14.0
ERROR: Ignored the following versions that require a different python version: 0.100.0 Requires-Python >=3.10,<3.13; 0.100.1 Requires-Python >=3.10,<3.13; 0.102.0 Requires-Python >=3.10,<3.13; 0.105.0 Requires-Python >=3.10,<3.13; 0.108.0 Requires-Python >=3.10,<3.13; 0.114.0 Requires-Python >=3.10,<3.13; 0.117.0 Requires-Python >=3.10,<3.13; 0.117.1 Requires-Python >=3.10,<3.13; 0.118.0 Requires-Python >=3.10,<3.13; 0.119.0 Requires-Python >=3.10,<3.13; 0.120.0 Requires-Python >=3.10,<3.13; 0.120.1 Requires-Python >=3.10,<3.13; 0.121.0 Requires-Python >=3.10,<3.13; 0.121.1 Requires-Python >=3.10,<3.13; 0.14.0 Requires-Python >=3.10,<=3.13; 0.14.0rc0 Requires-Python >=3.10,<3.12; 0.14.0rc1 Requires-Python >=3.10,<=3.13; 0.14.1 Requires-Python >=3.10,<=3.13; 0.14.3 Requires-Python >=3.10,<=3.13; 0.14.4 Requires-Python >=3.10,<=3.13; 0.16.0 Requires-Python >=3.10,<=3.13; 0.16.1 Requires-Python >=3.10,<=3.13; 0.

In [ ]:
#Monitoring 
import sys
!{sys.executable} -m pip install agentops

In [ ]:
# Install Google SDK into kernel environment (needed for Gemini to work) if using another provider skip this
import sys
!{sys.executable} -m pip install google-genai

In [ ]:
# Search Engine Tool for Agent 2
import sys
!{sys.executable} -m pip install tavily-python

In [47]:
# Scrape Tool for Agent 3
import sys
!{sys.executable} -m pip install scrapegraph-py

#### Getting a free Gemini API key

1. Go to [Google AI Studio](https://aistudio.google.com/) and sign in with any Google account.
2. Click **"Get API key"** (usually in the left sidebar or top right).
3. Click **"Create API key"** — you can create it in a new project or an existing Google Cloud project.
4. Copy the key and save it in your `.env` file as `GEMINI_API_KEY=your_key_here`.

### Imports & Load environment variables


In [49]:
import os 
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
import agentops

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List

from tavily import TavilyClient
from scrapegraph_py import client

In [50]:
# Load the variables from .env into the system environment
load_dotenv()

agentops_key = os.getenv("AGENTOPS_API_KEY")
gemini_key = os.getenv("GEMINI_API_KEY")
tavily_key = os.getenv("tavily_api")
scrapegraph_key = os.getenv("scrapegraph_api")

# Initialize AgentOps
agentops.init(
    api_key = os.getenv("AGENTOPS_API_KEY"),
    default_tags=('crewai')
)


In [ ]:
# create new repo for outputs 
output_dir = "./ai-agents-output"
os.makedirs(output_dir, exist_ok=True)

basic_llm = LLM(model="gemini/gemini-3.5-flash-lite", temperature=0)
tavily_client = TavilyClient(api_key=tavily_key)
scrapegraph_client = client(api_key=scrapegraph_key)

In [37]:
no_keywords = 10

## Setup Agents

### Agent A : suggested searh queries

In [38]:
class SuggestedSearchQueries(BaseModel):
    queries: List[str] = Field(...,title="Suggested Search Queries to be passed to the search engine",
                               min_items= 1, max_items=no_keywords)
    
search_queries_recommendation_agent = Agent(
    role = "Search Queries Recommendation Agent",
    goal = "\n".join([
        "to provide a list of suggested searh queries to be passed to the search engine.",
        "the queries must be varied and looking for specific items."
    ]),
    backstory = "The agent is designed to help in looking for product by providing a list of suggested search queries to be passed to the search engine based on the context provided.",
    llm = basic_llm,
    verbose = True,
)

search_queries_recommendation_task = Task(
    description= "\n".join([
        "rifland is looking to buy {product_name} at the best prices (value for a price strategy)",
        "the company target any of these websites to buy from : {websites_list}",
        "the company want to search all available products on the internet to be compared later in another stage",
        "The stores must sell the product in {contry_name}",
        "Generate only {no_keywords} queries",
        "The search query must reach an ecommerce webpage for product, and not a blog or listing page."
    ]),

    expected_output= "A JSON object containing a list of suggested search queries.",
    output_json = SuggestedSearchQueries,
    output_file=os.path.join(output_dir, "step_1_Suggested_Search_Queries.json"),
    agent=search_queries_recommendation_agent
)

C:\Users\Ha\AppData\Local\Temp\ipykernel_348\3585384287.py:2: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  queries: List[str] = Field(...,title="Suggested Search Queries to be passed to the search engine",
C:\Users\Ha\AppData\Local\Temp\ipykernel_348\3585384287.py:2: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  queries: List[str] = Field(...,title="Suggested Search Queries to be passed to the search engine",


### Agent B : search engine agent


In [44]:
class SingleSearchResults(BaseModel):
    title : str
    url : str
    content : str
    score : float
    search_query : str


class AllSearchResults(BaseModel):
    results : List[SingleSearchResults]
    
@tool
def search_engine_tool(query : str):
    """Useful for search-based queries. Use this to find current information about any query related pages using a search engine"""
    return tavily_client.search(query)

search_engine_agent = Agent(
    role = "Search Engine Agent ",
    goal = "to Search for products based on the suggested search query",
    backstory ="The agent is designed to help in looking for products by searching for products based on the suggested search queries ",
    llm = basic_llm,
    verbose = True,
    tools = [search_engine_tool]
)

search_engine_task =Task( 
    description= "\n".join([
        "The task is to search for products based on the suggested search queries.",
        "You have to collect results from multipe search queries.", 
        "Ignore any susbicious links or not an ecomerce single product website link.",
        "Ignore any search results with confidence score less than {score_th} .",
        "The search results will be used to compare prices of products from different websites."
    ]),
    expected_output="A JSON object containing the search results." ,
    output_json= AllSearchResults, 
    output_file=os.path.join(output_dir,"step_2_search results.json"),
    agent= search_engine_agent
)

## Run The Ai Crew

In [45]:
rifland_crew =Crew(
    agents= [
        search_queries_recommendation_agent,
        search_engine_agent
    ],
    tasks=[
        search_queries_recommendation_task,
        search_engine_task
    ],
    process =Process.sequential
)

In [46]:
crew_results = rifland_crew.kickoff( 
    inputs = {
        "product_name" : "coffe machine for office",
        "websites_list" : ["www.jumia.com.ma" ,"www.electroplanet.ma" ,"www.marjane.ma"],
        "contry_name" : "Morocco",
        "no_keywords" : 10,
        "score_th" : 0.20,
                      }
)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Search Queries Recommendation Agent                                                                     │
│                                                                                                                 │
│  Task: rifland is looking to buy coffe machine for office at the best prices (value for a price strategy)       │
│  the company target any of these websites to buy from : ['www.jumia.com.ma', 'www.electroplanet.ma',            │
│  'www.marjane.ma']                                                                                              │
│  the company want to search all available products on the internet to be compared later in another stage        │
│  The stores must sell the product in Morocco                                                                    │
│  Generate only 10 queries                                                                                       │
│  The search query must reach an ecommerce webpage for product, and not a blog or listing page.                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Search Queries Recommendation Agent                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  queries=['site:www.jumia.com.ma machine a cafe professionnelle', 'site:www.electroplanet.ma machine a cafe     │
│  bureau', 'site:www.marjane.ma cafetiere expresso', 'site:www.jumia.com.ma machine a cafe avec broyeur',        │
│  'site:www.electroplanet.ma cafetiere grain', 'site:www.marjane.ma machine a cafe programmable',                │
│  'site:www.jumia.com.ma cafetiere dosette pas cher', 'site:www.electroplanet.ma machine a cafe filtre',         │
│  'site:www.marjane.ma machine a cafe capsule', 'site:www.jumia.com.ma machine a cafe office']                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Search Engine Agent                                                                                     │
│                                                                                                                 │
│  Task: The task is to search for products based on the suggested search queries.                                │
│  You have to collect results from multipe search queries.                                                       │
│  Ignore any susbicious links or not an ecomerce single product website link.                                    │
│  Ignore any search results with confidence score less than 0.2 .                                                │
│  The search results will be used to compare prices of products from different websites.                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_engine_tool executed with result: {'query': 'machine a cafe professionnelle', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [], 'response_time': 1.24, 'request_id': 'bbb389d4-7148-4745-ae72-61abadbc2f6c'}...
Tool search_engine_tool executed with result: {'query': 'machine a cafe bureau', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.electroplanet.ma/p3076120-machine-a-cafe-avec-broyeur-1318-1400w-noir-ari...
Tool search_engine_tool executed with result: {'query': 'cafetiere expresso', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.marjane.ma/search/caf%C3%A9?auth=true', 'title': 'Recherche : café - Marjane...
Tool search_engine_tool executed with result: {'query': 'machine a cafe avec broyeur', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [], 'response_time': 1.29, 'request_id': '1a67ff28-fd2b-4ad5-b77b-da2cf02a3a71'}...
Tool search_eng

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Search Engine Agent                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  results=[SingleSearchResults(title='machine a cafe avec broyeur 1318 1400w noir ariet - Electroplanet',        │
│  url='https://www.electroplanet.ma/p3076120-machine-a-cafe-avec-broyeur-1318-1400w-noir-ariet.html',            │
│  content="Plus d'information. Largeur, 305 mm. Profondeur, 300 mm. Hauteur, 305 mm. Poids, 6,12 kg. Couleur du  │
│  produit, Noir. Type de produit, Machine à expresso.", score=0.26123476,                                        │
│  search_query='site:www.electroplanet.ma machine a cafe bureau'), SingleSearchResults(title='machine a cafe     │
│  ka5997 20 bars gris severin - Electroplanet',                                                                  │
│  url='https://www.electroplanet.ma/p3110149-machine-a-cafe-ka5997-20-bars-gris-severin.html', content="Plus     │
│  d'information. Puissance ; Puissance, 1350 W ; Type de produit, Machine à espresso ; Capacité en tasses, 2     │
│  tasses ; Capacité du réservoir d'eau, 1 L.", score=0.23914538, search_query='site:www.electroplanet.ma         │
│  machine a cafe bureau'), SingleSearchResults(title='MACHINE A CAFE ESPRESSO K5EC1-50MB 15BARS AEG',            │
│  url='https://www.electroplanet.ma/p2974134-aeg-machine-a-cafe-espresso-k5ec1.html', content="Plus              │
│  d'information. TYPE DE CAFE, MOULU. BUSE VAPEUR, Oui. Finition, NOIR. PROGRAMABLE, Oui. ARRET AUTOMATIQUE,     │
│  Oui. ECRAN, Oui. PLATEAU CHAUFFE TASSES, Oui.", score=0.221848, search_query='site:www.electroplanet.ma        │
│  machine a cafe bureau'), SingleSearchResults(title='Cafetière et machines expresso : Découvrez nos offres |    │
│  Electroplanet', url='https://www.electroplanet.ma/petit-electromenager/cafetiere-et-expresso', content='Chez   │
│  ElectroPlanet Maroc, découvrez une sélection des meilleures machines à café actuelles Delonghi, Krups, Sage,   │
│  Melitta, Philips, Moulinex… selon des critères', score=0.21922922, search_query='site:www.electroplanet.ma     │
│  machine a cafe bureau'), SingleSearchResults(title='Recherche : café - Marjane',                               │
│  url='https://www.marjane.ma/search/caf%C3%A9?auth=true', content='Machine à café espresso ECM-290 1400W 19     │
│  bars rouge - DENWA. 549,00DH 799,00DH', score=0.38269553, search_query='site:www.marjane.ma cafetiere          │
│  expresso'), SingleSearchResults(title='Cafetière et machines expresso : Découvrez nos offres |                 │
│  Electroplanet', url='https://www.electroplanet.ma/petit-electromenager/cafetiere-et-expresso', content='En     │
│  plus de différents modèles de machines principaux comme les machines à café grains et les machines expresso,   │
│  découvrez les cafetières filtres, les machines à', score=0.54521364, search_query='site:www.electroplanet.ma   │
│  cafetiere grain'), SingleSearchResults(title='MACHINE A CAFE ELX-CM-430 1,5L ELEXIA - Electroplanet',          │
│  url='https://www.electroplanet.ma/p3030005-machine-a-cafe-elx-cm-430-1-5l-elexia.html', content="Capable de    │
│  se remplir de 1,5L, conçue pour le café en grains, · Bearer d'environ 40,3 cm de hauteur, 23 cm de largeur et  │
│  45 cm de profondeur,", score=0.5363378, search_query='site:www.electroplanet.ma cafetiere grain'),             │
│  SingleSearchResults(title='Expresso avec broyeur à café - Electroplanet',                                      │
│  url='https://www.electroplanet.ma/petit-electromenager